# Shofly media worker (Colab)
Manual worker for authorized media. Optional aria2c is available only for explicitly allowlisted trusted hosts.
A mounted Google Drive folder can retain partial transfers across Colab restarts. Download and upload size limits remain unchanged.
Secrets belong in Colab Secrets, never in notebook cells.

## 1. Environment

In [ ]:
import os, subprocess, sys
# Clone Shofly once; subsequent runs update the existing checkout.
REPO_URL = "https://github.com/MohammedTeir/shofly-project.git"
BRANCH = "feat/aria2c-safe-downloader-v1"  # Draft testing only
REPO_DIR = "/content/shofly"
WORKER_DIR = f"{REPO_DIR}/workers/media-worker"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Existing repository found. Updating...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
elif os.path.exists(REPO_DIR):
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository. "
        "Inspect it before deleting or replacing it."
    )
else:
    print("Cloning Shofly repository...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

assert os.path.isdir(WORKER_DIR), "Media worker directory not found"
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "httpx==0.28.1", "pydantic-settings==2.7.0"], check=True)
if WORKER_DIR not in sys.path:
    sys.path.insert(0, WORKER_DIR)
print("Shofly repository ready!")
print("Worker directory:", WORKER_DIR)
print("ffprobe:", subprocess.run(["ffprobe", "-version"], capture_output=True, text=True).stdout.splitlines()[:1] or "MISSING (run: !apt-get -qq install -y ffmpeg)")
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], check=True)
print("aria2c:", subprocess.run(["aria2c", "--version"], capture_output=True, text=True, check=True).stdout.splitlines()[0])


## 2. Optional trusted-source aria2c setup
Choose HTTPX for ordinary sources. For authorized media hosted on a domain you control, set a trusted host and enable aria2c. Drive storage is mounted only in aria2c mode.

In [ ]:
from pathlib import Path
# Configure before loading WorkerSettings. Use only trusted, authorized source hosts.
USE_ARIA2 = True
TRUSTED_SOURCE_HOSTS = ""  # REQUIRED for aria2c: comma-separated domains you trust
if USE_ARIA2:
    if not TRUSTED_SOURCE_HOSTS.strip():
        raise ValueError("Set TRUSTED_SOURCE_HOSTS to your trusted source domains before enabling aria2c.")
    from google.colab import drive
    drive.mount("/content/drive")
    storage = Path("/content/drive/MyDrive/Shofly/downloads")
    storage.mkdir(parents=True, exist_ok=True)
    os.environ["DOWNLOADER"] = "aria2c"
    os.environ["ALLOWED_HOSTS"] = TRUSTED_SOURCE_HOSTS
    os.environ["PERSISTENT_DOWNLOAD_DIR"] = str(storage)
    os.environ["ARIA2_CONNECTIONS"] = "4"
    os.environ["ARIA2_RETRIES"] = "3"
    print("Persistent transfer storage ready:", storage)
else:
    os.environ["DOWNLOADER"] = "httpx"
    os.environ.pop("PERSISTENT_DOWNLOAD_DIR", None)
    print("HTTPX downloader selected.")

## 2. Secrets and checks

In [ ]:
from google.colab import userdata
os.environ["API_BASE_URL"] = userdata.get("SHOFLY_API_URL")
os.environ["WORKER_API_TOKEN"] = userdata.get("WORKER_API_TOKEN")
os.environ["PROVIDER"] = "telegram"   # or "local" for a dry run into /content/stored
if os.environ["PROVIDER"] == "telegram":
    os.environ["TELEGRAM_BOT_TOKEN"] = userdata.get("TELEGRAM_BOT_TOKEN")
    os.environ["TELEGRAM_CHAT_ID"] = userdata.get("TELEGRAM_STORAGE_CHANNEL_ID")

import httpx
from shofly_worker.config import WorkerSettings
from shofly_worker.client import BackendClient
from shofly_worker.providers import LocalDirProvider, TelegramProvider
S = WorkerSettings()
client = BackendClient(S.api_base_url, S.worker_api_token, S.worker_id)
print("backend live:", httpx.get(S.api_base_url + "/health/live", timeout=15).json())
print("worker token accepted:", client.ping(), "| worker id:", S.worker_id)
provider = TelegramProvider(S.telegram_bot_token, S.telegram_chat_id, S.max_bytes) if S.provider == "telegram" else LocalDirProvider(S.local_dir)

## 3. Process jobs
Prints one outcome per job (no URLs or tokens). Stop the cell at any time: unfinished jobs are re-queued when their lease expires.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)  # avoid logging signed download URLs
from shofly_worker.runner import run_until_empty
dl_http = httpx.Client(timeout=60.0)
results = run_until_empty(client, provider, S, dl_http)
from collections import Counter
print("processed:", len(results), dict(Counter(results)))

## 4. Cleanup

In [ ]:
import shutil
# Only clean ephemeral temp files. Persistent files are managed by retention.py.
shutil.rmtree(S.tmp_dir, ignore_errors=True)
print("Ephemeral temp files removed; persistent storage untouched.")